In [1]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import ray
import scipy
import torch

from metrics import metrics, losses
from experiment import experiment
import model_interpretation

## Configuration and Loading

In [2]:
device = torch.device("cuda:0")

In [3]:
params_path = "trained_models/AlphaGenome/params.json"
checkpoint_path = "trained_models/AlphaGenome/checkpoint_000199.pt"
test_csv = "data/distillation_test.csv.gz"
activation = "sigmoid"
num_positions = 120

In [4]:
with open(params_path) as params_file:
    params = json.load(params_file)
    params["skip_dataset"] = True
    params["skip_dataloader"] = True
    params["skip_compiling"] = True
    params["device"] = device

if activation == "identity":
    activation_fn = lambda x: x
elif activation == "sigmoid":
    activation_fn = lambda x: scipy.special.expit(x)
else:
    raise ValueError("Activation function was not valid")

In [5]:
exp = experiment.Experiment(params)

compiled_ckpt = torch.load(checkpoint_path, map_location=device)
compiled_state_dict = compiled_ckpt["model_state_dict"]
cleaned_state_dict = {
    k.replace("_orig_mod.", "", 1): v for k, v in compiled_state_dict.items()
}
exp.model.load_state_dict(cleaned_state_dict)
exp.model = exp.model.to(device).eval()

/home/coder/.local/lib/python3.12/site-packages/ray/tune/trainable/trainable.py:688: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  self._result_logger = UnifiedLogger(config, self._logdir, loggers=None)
/home/coder/.local/lib/python3.12/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
The `JsonLogger interface is deprecated in favor of the `ray.tune.json.JsonLoggerCallback` interface and will be removed in Ray 2.7.
  self._loggers.append(cls(self.config, self.logdir, self.trial))
/home/coder/.local/lib/python3.12/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be remov

Loading model
Sending model to device


## Evaluate Distillation Performance

In [6]:
test_df = pd.read_csv(test_csv)

In [7]:
testlabels, testpreds = model_interpretation.get_model_predictions(
    params, exp.model, test_df, device, activation_fn
)

  0%|          | 0/45 [00:00<?, ?it/s]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(2.25,2.1), dpi=600)
model_interpretation.plot_hex_histogram(
    fig,
    ax,
    x=testpreds,
    y=testlabels,
    gridsize=50
)
ax.set_yticks([0, 0.5, 1.0])
ax.set_xticks([0, 0.5, 1.0])
ax.set_xlabel("AlphaGenome")
ax.set_ylabel("Distilled Model")
ax.set_title("")
fig.savefig("figures/AlphaGenome_Distillation_Performance.svg", dpi=600)

## Plot Sequence Features

In [9]:
with torch.no_grad():
    kmer_generator = model_interpretation._random_kmers_generator(
        ["A", "C", "G", "T"],
        exp.model.seq_conv.convs["0"].kernel_size[0],
        10_000_000,
        42,
    )

    kmers, seq_activations = model_interpretation.compute_conv1d_activations_batched(
        conv_layer=exp.model.seq_conv.convs["0"],
        activation_layer=exp.model.seq_conv.activation,
        batchnorm_layer=exp.model.seq_conv.batch_norm,
        alphabet=["A", "C", "G", "T"],
        device=device,
        kmer_generator=kmer_generator,
        batch_size=2048,
    )

0it [00:00, ?it/s]

In [10]:
with torch.no_grad():
    kmer_contribution_df, linspace_contribution_df = (
        model_interpretation.calculate_first_order_shape_functions(
            torch.tensor(seq_activations), exp.model, 0, num_positions, device
        )
    )

In [11]:
## Rank feature importance by ablation

if activation == "sigmoid":
    metric_fns = {
        "pearson": metrics.METRICS["pearson-logits-r"],
        "kl-divergence": losses.LOSSES["kl_divergence_logits"],
    }
else:
    metric_fns = {
        "pearson": metrics.METRICS["pearson-r"],
        "mse": losses.LOSSES["mse"],
    }
    
feature_ablations = model_interpretation.calculate_feature_nam_ablations(
    params, exp.model, test_df, exp.model.seq_conv.convs["0"].out_channels, 
    metric_fns, device, activation_fn
)

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

...

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

In [12]:
conv_kernel_ordering = sorted(
    feature_ablations.keys(), key = lambda idx: feature_ablations[idx]["pearson"], reverse=False
)

In [15]:
for i in range(exp.model.seq_conv.convs["0"].out_channels):        
    fig = model_interpretation.plot_kernel(
        exp.model.seq_conv.convs["0"].weight,
        kmers,
        seq_activations,
        linspace_contribution_df.drop_duplicates(ignore_index=True),
        seq_activations.shape[1],
        num_positions,
        i,
        logo_q=[0.95, 1.0],
        conv_kernel_ordering=conv_kernel_ordering,
        label=f"{i + 1}",
        figsize=(1.25,1.0),
        fontsize=6,
    )

    # Remove figure background
    fig.patch.set_visible(False)

    fig.savefig(
        f"figures/AlphaGenome_Distillation_Feature_{i}.svg", 
        dpi=600, 
        bbox_inches="tight"
    )
    plt.close()

/machine/GitHub/splicing-interpretable-distillation/distillation/model_interpretation.py:287: UserWarning: All k-mer activations were zero. Sequence logo will be empty.
  warnings.warn("All k-mer activations were zero. Sequence logo will be empty.")


## Plot Contribution Sum -> Prediction Function 

In [16]:
tuner_domain, output_samples = model_interpretation.get_tuner_domain(exp.model, test_df, params, device, 0)

  0%|          | 0/1426 [00:00<?, ?it/s]

In [ ]:
# Plot tuner function and histogram
fig = model_interpretation.plot_tuner(
    exp.model.tuners["0"],
    f"",
    domain=tuner_domain,
    output_samples=output_samples,
    device=device,
)

fig.savefig(
    f"figures/AlphaGenome_Distillation_PredictionFunction.svg"
)